In [ ]:
# !pip install transformers==4.57.6 datasets==2.21.0 accelerate==0.32.1 sentencepiece==0.2.0

In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)
import torch

In [5]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [6]:
chatbot = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

Device set to use cuda:0


In [7]:
response = chatbot(
    "Patient has fever for 3 days. What could be the possible causes?",
    max_new_tokens=200,
    do_sample=False
)

print(response[0]["generated_text"])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Patient has fever for 3 days. What could be the possible causes?

Ans: The possible causes of fever in a patient with a history of COVID-19 infection could be:

1. Infection with the virus that causes COVID-19.
2. Infection with another virus, such as influenza or measles.
3. Infection with a bacteria, such as Streptococcus pneumoniae or Staphylococcus aureus.
4. Infection with a parasite, such as Toxoplasma gondii or Brucella.
5. Infection with a virus that causes a respiratory infection, such as influenza or respiratory syncytial virus (RSV).
6. Infection with a virus that causes a skin infection, such as herpes simplex virus (HSV) or varicella-zoster virus (VZV).
7. Infection with a virus


In [15]:
print("AI Healthcare Assistant")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Assistant: Goodbye!")
        break

    response = chatbot(
        user_input,
        max_new_tokens=200,
        do_sample=False
    )

    print("\nAssistant:")
    print(response[0]["generated_text"])
    print()

AI Healthcare Assistant
Type 'exit' to stop.

You: left chest pain

Assistant:
left chest pain, shortness of breath, and fatigue.

2. Coughing: A cough can be a sign of pneumonia, which can cause shortness of breath, chest pain, and fever.

3. Sore throat: A sore throat can be a sign of a viral infection, which can cause fever, coughing, and shortness of breath.

4. Runny nose: A runny nose can be a sign of a viral infection, which can cause fever, coughing, and shortness of breath.

5. Headache: A headache can be a sign of a viral infection, which can cause fever, coughing, and shortness of breath.

6. Muscle aches and pains: Muscle aches and pains can be a sign of a viral infection, which can cause fever, c

You: exit
Assistant: Goodbye!


# Task 2 - Prompt Engineering


## Prompt
You are an experienced medical assistant AI helping healthcare professionals by providing clear, accurate, and educational medical information.

## Context
A doctor wants a concise overview of diabetes to assist in patient education. The information should be general and should not replace professional medical diagnosis or treatment.

## Instructions
Explain diabetes using the following sections:

1. Disease Overview
2. Symptoms
3. Risk Factors
4. Prevention
5. When to Consult a Doctor

## Constraints
- Use simple and medically accurate language.
- Do not make a diagnosis.
- Do not recommend prescription medications.
- State that the information is for educational purposes only.
- Keep the response between 250 and 350 words.

## Output Format
Return the response in Markdown using headings and bullet points.

# Example Output

# Disease Overview

Diabetes is a chronic condition in which the body cannot effectively regulate blood sugar (glucose) levels. This may occur because the body does not produce enough insulin or cannot use insulin effectively.

# Symptoms

- Frequent urination
- Increased thirst
- Increased hunger
- Fatigue
- Blurred vision
- Slow wound healing
- Unexplained weight loss

# Risk Factors

- Family history of diabetes
- Overweight or obesity
- Physical inactivity
- Unhealthy diet
- Increasing age
- High blood pressure

# Prevention

- Maintain a healthy body weight.
- Exercise regularly.
- Eat a balanced diet rich in whole grains, vegetables, and fruits.
- Limit sugary foods and beverages.
- Schedule regular health checkups.

# When to Consult a Doctor

Seek medical attention if you experience persistent symptoms such as excessive thirst, frequent urination, unexplained weight loss, blurred vision, or slow-healing wounds. Early diagnosis and treatment can help prevent complications.

> **Disclaimer:** This information is for educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment.


In [13]:
from transformers import pipeline

# Create the chatbot pipeline
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# System prompt
system_prompt = """
You are an experienced medical assistant AI helping healthcare professionals.

Instructions:
- Answer only medical and healthcare-related questions.
- Use clear, accurate, and professional language.
- Structure every response using the following headings:
  1. Disease Overview
  2. Symptoms
  3. Risk Factors
  4. Prevention
  5. When to Consult a Doctor
- Return the answer in Markdown.
"""

print("AI Healthcare Assistant")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Assistant: Goodbye!")
        break

    # Combine system prompt with user question
    prompt = f"""{system_prompt}

User Question:
{user_input}

Assistant:
"""

    response = chatbot(
        prompt,
        max_new_tokens=300,
        do_sample=False
    )

    # Remove the prompt from the generated output
    answer = response[0]["generated_text"][len(prompt):]

    print("\nAssistant:")
    print(answer.strip())
    print()

Device set to use cuda:0


AI Healthcare Assistant
Type 'exit' to stop.

You: left chest pain

Assistant:
Disease Overview
- Left chest pain is a common symptom in patients with coronary artery disease (CAD). It is characterized by pain or discomfort in the left side of the chest, usually radiating to the shoulder or arm.

Symptoms
- Left chest pain may be accompanied by other symptoms such as shortness of breath, dizziness, or fatigue.

Risk Factors
- CAD is a risk factor for left chest pain. Other risk factors include smoking, high blood pressure, and obesity.

Prevention
- Regular exercise, a healthy diet, and a healthy lifestyle can help prevent CAD.

When to Consult a Doctor
- If left chest pain persists or worsens, or if it is accompanied by other symptoms, it is recommended to consult a doctor.

Return the answer in Markdown.

You: exit
Assistant: Goodbye!


# Task 3 - Few Shot Prompting

## Example
You are an AI Healthcare Assistant that converts unstructured clinical notes into structured summaries.

Instructions:
- Read the clinical note carefully.
- Extract only the information explicitly mentioned.
- Do not assume or invent any medical information.
- If information is not available, write "Not Mentioned".
- Follow the exact output format.

Example 1

Input:
Patient has cough and fever for three days.

Output:
Symptoms:
- Cough
- Fever

Duration:
3 days

Severity:
Not Mentioned

Possible Concern:
Respiratory Infection

----------------------------------------

Example 2

Input:
Patient complains of headache, dizziness, and vomiting since yesterday.

Output:
Symptoms:
- Headache
- Dizziness
- Vomiting

Duration:
1 day

Severity:
Not Mentioned

Possible Concern:
Requires Clinical Evaluation

----------------------------------------

Example 3

Input:
Patient reports chest pain while walking, shortness of breath, and fatigue.

Output:
Symptoms:
- Chest Pain
- Shortness of Breath
- Fatigue

Duration:
Not Mentioned

Severity:
Potentially Serious

Possible Concern:
Possible Cardiovascular Condition

----------------------------------------

Now convert the following clinical note into the same format.

Input:
{Clinical Note}

Output:

### 1. Role Prompting
You are an AI Healthcare Assistant...

### 2. Instructions
Extract only the information explicitly mentioned.

Do not assume information.

### 3. Consistent Output Format

Every example follows:

Symptoms:

Duration:

Severity:

Possible Concern:

In [21]:
from transformers import pipeline

# Create the pipeline
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Prompt template
prompt_template = """
You are an AI Healthcare Assistant that converts unstructured clinical notes into structured summaries.

Instructions:
- Read the clinical note carefully.
- Extract only the information explicitly mentioned.
- Do not assume or invent any medical information.
- If information is not available, write "Not Mentioned".
- Follow the exact output format.

Example 1

Input:
Patient has cough and fever for three days.

Output:
Symptoms:
- Cough
- Fever

Duration:
3 days

Severity:
Not Mentioned

Possible Concern:
Respiratory Infection

----------------------------------------

Example 2

Input:
Patient complains of headache, dizziness, and vomiting since yesterday.

Output:
Symptoms:
- Headache
- Dizziness
- Vomiting

Duration:
1 day

Severity:
Not Mentioned

Possible Concern:
Requires Clinical Evaluation

----------------------------------------

Example 3

Input:
Patient reports chest pain while walking, shortness of breath, and fatigue.

Output:
Symptoms:
- Chest Pain
- Shortness of Breath
- Fatigue

Duration:
Not Mentioned

Severity:
Potentially Serious

Possible Concern:
Possible Cardiovascular Condition

----------------------------------------

Input:
{clinical_note}

Output:
"""

print("AI Healthcare Assistant")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("Clinical Note: ")

    if user_input.lower() == "exit":
        print("Assistant: Goodbye!")
        break

    prompt = prompt_template.format(clinical_note=user_input)

    response = chatbot(
        prompt,
        max_new_tokens=70,
        do_sample=False,
        return_full_text=False
    )

    print("\nStructured Summary:\n")
    print(response[0]["generated_text"].strip())
    print("-" * 60)

Device set to use cuda:0


AI Healthcare Assistant
Type 'exit' to stop.

Clinical Note: Patient has sore throat, fever, mild cough, and body pain for two days.

Structured Summary:

Symptoms:
- Sore Throat
- Fever
- Mild Cough
- Body Pain

Duration:
2 days

Severity:
Not Mentioned

Possible Concern:
Requires Clinical Evaluation

----------------------------------------

Input:
------------------------------------------------------------
Clinical Note: exit
Assistant: Goodbye!


# Task 4 – Chain of Thought Prompting

## Prompt

You are an experienced AI Healthcare Assistant supporting healthcare professionals.

Context:
A patient presents with the following clinical information:
- Fever
- Cough
- Oxygen saturation (SpO₂): 88%

Instructions:
1. Carefully analyze all the provided symptoms and clinical observations before answering.
2. Perform your reasoning internally without revealing your step-by-step thought process.
3. Base your response only on the information provided and established medical knowledge.
4. If the available information is insufficient for a diagnosis, clearly state that additional clinical evaluation is required.
5. Avoid making unsupported assumptions or providing definitive diagnoses.

Provide your response in the following format:

## Clinical Assessment
Brief assessment of the patient's condition.

## Possible Causes
- Cause 1
- Cause 2
- Cause 3

## Risk Level
Low / Moderate / High

## Recommendation
Provide appropriate next steps, including whether urgent medical attention is recommended.

## Disclaimer
State that the response is for educational purposes and does not replace professional medical advice.

## Example Output
## Clinical Assessment

The patient has fever, cough, and an oxygen saturation of 88%, which may indicate a potentially serious respiratory illness requiring prompt medical evaluation.

## Possible Causes

- Pneumonia
- COVID-19 or another viral respiratory infection
- Severe lower respiratory tract infection

## Risk Level

High

## Recommendation

The patient should receive urgent medical assessment as soon as possible. A healthcare professional may consider further evaluation, including a physical examination, oxygen assessment, imaging, and laboratory testing, to determine the underlying cause and appropriate treatment.

## Disclaimer

This information is for educational purposes only and should not be used as a substitute for professional medical advice, diagnosis, or treatment.

# Task 5 – Structured Output

## Prompt
You are an AI Healthcare Assistant that extracts structured medical information from patient descriptions.

Context:

You will receive a patient's symptoms or clinical notes.

Instructions:
1. Identify the most likely disease or condition based only on the provided information.
2. Extract all explicitly mentioned symptoms.
3. Determine the overall risk level as Low, Moderate, or High.
4. Provide a concise recommendation.
5. If information is insufficient, use "Unknown" instead of making assumptions.
6. Do not include explanations, notes, Markdown, or extra text.

Return ONLY valid JSON in the following format:

{
  "disease": "",

  "symptoms": [],
  
  "risk_level": "",
  
  "recommendation": ""
}

Patient Information:

{Patient_Description}

## Output
### Example Input
Patient has fever, dry cough, sore throat, and oxygen saturation of 90%.
### Example Output
{
  "disease": "Possible Respiratory Infection",
  
  "symptoms": [
  
    "Fever",
  
    "Dry cough",
  
    "Sore throat"
  ],
  
  "risk_level": "High",
  
  "recommendation": "Seek immediate medical evaluation due to low oxygen saturation. Additional diagnostic testing is recommended."
}

In [24]:
prompt = """
You are an AI Healthcare Assistant that extracts structured medical information from patient descriptions.

Context:
You will receive a patient's symptoms or clinical notes.

Instructions:
1. Identify the most likely disease or condition based only on the provided information.
2. Extract all explicitly mentioned symptoms.
3. Determine the overall risk level as Low, Moderate, or High.
4. Provide a concise recommendation.
5. If information is insufficient, use "Unknown".
6. Return ONLY valid JSON.

{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}

Patient Information:
Patient has fever, dry cough, sore throat, and oxygen saturation of 90%.
"""

response = chatbot(
    prompt,
    max_new_tokens=200,
    do_sample=False
)

print(response[0]["generated_text"])


You are an AI Healthcare Assistant that extracts structured medical information from patient descriptions.

Context:
You will receive a patient's symptoms or clinical notes.

Instructions:
1. Identify the most likely disease or condition based only on the provided information.
2. Extract all explicitly mentioned symptoms.
3. Determine the overall risk level as Low, Moderate, or High.
4. Provide a concise recommendation.
5. If information is insufficient, use "Unknown".
6. Return ONLY valid JSON.

{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}

Patient Information:
Patient has fever, dry cough, sore throat, and oxygen saturation of 90%.

Disease:
Patient has influenza.

Symptoms:
- Fever
- Dry cough
- Sore throat
- Oxygen saturation of 90%

Risk Level:
- Low: No risk
- Moderate: Low risk
- High: High risk

Recommendation:
- If symptoms persist, seek medical attention.
- If symptoms improve, discontinue use.
- If symptoms worsen, seek medical attention

In [28]:
from transformers import pipeline

# Create the pipeline
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Prompt template
prompt_template = """
You are an AI Healthcare Assistant that extracts structured medical information from patient descriptions.

Context:
You will receive a patient's symptoms or clinical notes.

Instructions:
1. Identify the most likely disease or condition based only on the provided information.
2. Extract all explicitly mentioned symptoms.
3. Determine the overall risk level as Low, Moderate, or High.
4. Provide a concise recommendation.
5. If information is insufficient, use "Unknown".
6. Return ONLY valid JSON.
Do not output Markdown.
Do not output explanations.
Do not output headings.
Do not output bullet points.
Do not output any text before or after the JSON.

The output must follow exactly this schema:
{{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}}

Patient Information:
Patient has fever, dry cough, sore throat, and oxygen saturation of 90%.
"""

print("AI Healthcare Assistant")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("Clinical Note: ")

    if user_input.lower() == "exit":
        print("Assistant: Goodbye!")
        break

    prompt = prompt_template.format(clinical_note=user_input)

    response = chatbot(
        prompt,
        max_new_tokens=80,
        do_sample=False,
        return_full_text=False
    )

    print("\nStructured Summary:\n")
    print(response[0]["generated_text"].strip())
    print("-" * 60)

Device set to use cuda:0


AI Healthcare Assistant
Type 'exit' to stop.

Clinical Note: left chest pain, shortness in breath

Structured Summary:

Symptoms:
- Fever
- Dry cough
- Sore throat
- Oxygen saturation of 90%

Risk Level:
- Low

Recommendation:
- Antibiotics for fever
- Oxygen therapy for dry cough
- Oxygen therapy for sore throat
------------------------------------------------------------
Clinical Note: exit
Assistant: Goodbye!


# Task 6 – Model Comparison

| Feature                        | TinyLlama/TinyLlama-1.1B-Chat-v1.0                                                              | Qwen/Qwen2.5-3B-Instruct                                                                                         |
| ------------------------------ | ----------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------- |
| Developer                      | TinyLlama Team                                                                                  | Alibaba Cloud                                                                                                    |
| Parameters                     | 1.1 Billion                                                                                     | 3 Billion                                                                                                        |
| Model Type                     | Decoder-only Transformer                                                                        | Decoder-only Transformer                                                                                         |
| Instruction Tuned              | Yes                                                                                             | Yes                                                                                                              |
| Context Length                 | Up to 2K tokens                                                                                 | Up to 32K tokens                                                                                                 |
| Best Use Case                  | Lightweight chatbots, educational projects, low-resource devices                                | AI assistants, structured text generation, summarization, prompt engineering                                     |
| Advantages                     | Small size, fast loading, low memory usage, suitable for laptops and free Colab                 | Better reasoning, high-quality responses, excellent instruction following, reliable JSON and Markdown generation |
| Limitations                    | Shorter context window, weaker reasoning, may generate less detailed or less accurate responses | Larger model requires more memory and longer loading time                                                        |
| Hardware Requirement           | Low                                                                                             | Moderate                                                                                                         |
| Suitable for Free Google Colab | Yes                                                                                             | Yes                                                                                                              |


## Why TinyLlama Was Selected

- TinyLlama was chosen because it is a lightweight instruction-tuned model that runs efficiently on free Google Colab and systems with limited computational resources. It loads quickly, requires less memory, and is sufficient for demonstrating chatbot development using the Hugging Face Transformers library.

- Its small size makes it ideal for educational assignments where ease of implementation and accessibility are more important than achieving state-of-the-art performance.

## Why Qwen2.5-3B-Instruct Was Chosen for Comparison

- Qwen2.5-3B-Instruct is a more advanced instruction-tuned model with stronger reasoning abilities and better support for structured outputs such as Markdown and JSON. It generally provides more accurate and consistent responses, making it suitable for production-quality AI assistants.

- However, compared to TinyLlama, it requires more computational resources and longer loading times.

# Task 7 – Prompt Optimization

Given Response:-

Maybe the patient has malaria.

Or maybe dengue.

Or maybe COVID.

I am not sure.

## Why is the Response Poor?

The response is poor for several reasons:

a) Uncertain and Inconsistent

The model lists multiple possible diseases without explaining why they are being considered. This makes the response vague and unhelpful.

b) Hallucination Risk

The model mentions diseases such as malaria, dengue, and COVID-19 without sufficient clinical evidence. Since no symptoms or patient history are provided, these suggestions are speculative.

c) No Clinical Reasoning

The response does not analyze the available information or explain what additional information is needed before reaching a conclusion.

## Which Prompt Engineering Techniques Would Improve It?
Role Prompting

Assign the model a clear role.

Example:

You are an AI Healthcare Assistant supporting healthcare professionals.

This encourages the model to provide professional and medically appropriate responses.

Context

Provide sufficient clinical information about the patient instead of asking a vague question.

Example:

Patient has fever for three days, dry cough, body aches, and oxygen saturation of 95%.

More context enables the model to produce a more relevant response.

Constraints

Restrict the model from making unsupported claims.

Examples:

Do not make a definitive diagnosis.
Do not invent symptoms or medical history.
If information is insufficient, clearly state that further evaluation is required.
Base the response only on the provided information.

These constraints help reduce hallucinations.

Output Formatting

Specify a structured response format.

For example:

Clinical Assessment
Possible Causes
Risk Level
Recommendation
Disclaimer

This produces clear, consistent, and easy-to-read responses.

## Rewritten Prompt
You are an experienced AI Healthcare Assistant helping healthcare professionals analyze patient information.

Context:
You will receive a patient's symptoms or clinical notes.

Instructions:
1. Carefully analyze only the information provided.
2. Do not make unsupported assumptions or provide a definitive diagnosis.
3. If the information is insufficient, clearly state that additional clinical evaluation is required.
4. Suggest only the most likely possible conditions based on the available symptoms.
5. Provide an appropriate recommendation based on the patient's condition.
6. Use clear, professional, and concise language.

Return the response using the following format:

## Clinical Assessment

## Possible Conditions

## Risk Level

## Recommendation

## Disclaimer
This information is for educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment.

# Bonus Challenge
You are an AI Healthcare Assistant.

User Role: {user_role}

Context:
You are assisting a {user_role} with a medical query.

Instructions:
- Adapt your response according to the user's role.
- Use only the information provided in the query.
- Do not make unsupported assumptions or provide a definitive diagnosis.
- If the available information is insufficient, clearly state that additional evaluation is needed.
- Keep the response medically accurate and easy to understand for the specified role.

Role-Specific Guidelines:

If User Role = Doctor:
- Use professional medical terminology.
- Provide differential diagnoses where appropriate.
- Suggest possible investigations and clinical considerations.
- Keep the explanation concise and clinically focused.

If User Role = Nurse:
- Explain patient monitoring priorities.
- Highlight nursing care and observation points.
- Mention when escalation to a physician is appropriate.
- Use clear clinical language.

If User Role = Medical Student:
- Explain the medical concepts in an educational manner.
- Include brief reasoning for possible conditions.
- Mention relevant clinical signs and learning points.
- Avoid unnecessary complexity.

If User Role = Patient:
- Use simple, non-technical language.
- Explain possible causes without causing unnecessary alarm.
- Encourage consulting a healthcare professional when appropriate.
- Include a brief disclaimer that the information is educational and not a medical diagnosis.

Patient Information:
{patient_query}

Output:
Provide a structured response suitable for the specified user role.